# Offline ALNS Repair Model Training — Improved Pipeline

**Colab-ready** | Explicit seeds | Grouped repair decisions | Dataset integrity checks | Ranking quality gates | Covariate-shift mitigation

This notebook runs the full offline training pipeline for the Hybrid ALNS repair model.
It is the authoritative, executable version of the improvements documented in `README.md`.

## Pipeline overview

```
generate grouped BFD-prefix data (v1) ──► train baseline (v1) ──► collect grouped ALNS states
                                                                               │
                           generate grouped BFD-prefix data (v2) ──────────────┘
                                           │
                                  train improved model (v2) ← quality gates checked here
```

## Improvement areas addressed

| Area | What changed |
|---|---|
| **Reproducibility** | All seeds defined once as constants; passed to every step |
| **Grouped repair data** | Synthetic data is generated from reachable BFD-prefix states and stores one `groups` id per placement decision |
| **Dataset integrity** | Feature version, feature count, label values, `groups` shape, and NaN/inf values checked on load |
| **Grouped validation** | Grouped train/test split and grouped CV prevent candidate leakage between train and test |
| **Ranking quality gates** | ROC-AUC, Average Precision, and top-1 repair accuracy checked on holdout |
| **Covariate shift** | ALNS states required for v2+ (`require_alns_states=True`); proportion reported |


## Cell 1 — Colab / local setup

Detects whether the notebook is running on Google Colab or locally.
On Colab, it prompts you to upload the repository zip and installs it.
Locally, it walks up from the current directory to find the repo root.

> **Colab tip**: if this is your first run, upload `bin-packing-optimization.zip`
> when the file picker appears. Subsequent runs can skip the upload if the
> `/content` directory still has the extracted repo.


In [ ]:
import os
import sys
import zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def find_repo_root(start: str) -> Path:
    """Walk up from `start` until a directory containing both
    pyproject.toml and requirements.txt is found."""
    for root, _, files in os.walk(start):
        if "pyproject.toml" in files and "requirements.txt" in files:
            return Path(root)
    raise FileNotFoundError("Could not find repo root with pyproject.toml")


repo_root = None
if IN_COLAB:
    try:
        repo_root = find_repo_root("/content")
    except FileNotFoundError:
        from google.colab import files

        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("Please upload the repo zip to continue.")
        zip_name = next(iter(uploaded))
        with zipfile.ZipFile(zip_name, "r") as zip_ref:
            zip_ref.extractall("/content")
        repo_root = find_repo_root("/content")
else:
    repo_root = find_repo_root(os.getcwd())

os.chdir(repo_root)
print("Repo root:", repo_root)


## Cell 2 — Install dependencies

Installs Python dependencies from `requirements.txt` and registers the repo
as an editable package so all internal imports resolve correctly.
The `-q` flag suppresses verbose pip output to keep the notebook readable.


In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .


## Cell 3 — Seed constants and path setup

All randomness is controlled by four constants defined here.
Using separate seeds per step allows independent auditing and partial reruns
without affecting other steps.

| Constant | Used by | Purpose |
|---|---|---|
| `SEED` | `train_repair_model` | Train/test split + GradientBoosting `random_state` |
| `SYNTHETIC_V1_SEED` | `generate_dataset` (v1) | Baseline synthetic dataset |
| `ALNS_SEED` | `collect_alns_states` | ALNS rollout RNG |
| `SYNTHETIC_V2_SEED` | `generate_dataset` (v2) | Supplementary synthetic dataset |


In [ ]:
import random
import numpy as np
import sys
import os
from pathlib import Path

# ── Reproducibility: all seeds defined here, passed to every downstream call ──
SEED              = 42   # train/test split + model random_state
SYNTHETIC_V1_SEED = 0    # generate_dataset v1
ALNS_SEED         = 1    # collect_alns_states
SYNTHETIC_V2_SEED = 2    # generate_dataset v2

random.seed(SEED)
np.random.seed(SEED)

# Ensure the repo root is in sys.path so the package is importable
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

TRAINING_DIR = (
    Path(repo_root)
    / "bin_packing_optimization"
    / "hybrid_learning_metaheuristics"
    / "hybrid_alns"
    / "repair_model_training"
)
DATA_DIR = TRAINING_DIR / "training_data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Change to the training directory so relative paths in config work correctly
os.chdir(TRAINING_DIR)
print("Training dir :", TRAINING_DIR)
print("Data dir     :", DATA_DIR)
print(f"Seeds        : SEED={SEED}, V1={SYNTHETIC_V1_SEED}, ALNS={ALNS_SEED}, V2={SYNTHETIC_V2_SEED}")

# Now imports should work correctly
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.repair_model_training.generate_dataset import (
    GenerateDatasetConfig,
    generate_dataset,
)
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.repair_model_training.collect_alns_states import (
    CollectAlnsStatesConfig,
    collect_alns_states,
)
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.repair_model_training.train_repair_model import (
    TrainRepairModelConfig,
    train_repair_model,
)

## Step 1 — Generate baseline synthetic dataset

Generates `synthetic_v1.pkl`: 4 000 random bin-packing instances with items drawn
from uniform, bimodal, and Gaussian distributions.

For each instance, `generate_dataset` now replays **BFD incrementally**.  At each
placement decision it labels the feasible existing bins from the current partial
packing before inserting the item.  The minimum-slack bin receives label `1`,
sampled alternative feasible bins receive label `0`, and all candidates from the
same placement decision share one `groups` id.

This is important because the repair model is used as a ranker at runtime: for one
displaced item, it scores all feasible bins and the solver chooses the argmax.
The grouped dataset lets training and evaluation preserve that decision structure.

**Expected output** (printed by `generate_dataset`):
- Total rows, positive rate, class counts.
- A saved payload containing `X`, `y`, `groups`, `feature_version`, `source`, and `summary`.
- Shape and NaN/inf integrity checks before saving.


In [ ]:
generate_dataset(
    GenerateDatasetConfig(
        instances=4000,
        n_min=50,
        n_max=200,
        max_negatives=5,
        seed=SYNTHETIC_V1_SEED,
        workers=1,          # set > 1 for speed on multi-core Colab runtimes
        output=str(DATA_DIR / "synthetic_v1.pkl"),
    )
)


## Step 2 — Train baseline model (v1)

Trains a `GradientBoostingClassifier` on `synthetic_v1.pkl` only.
This is the v1 baseline — no ALNS states are required yet
(`require_alns_states=False`).

**What this cell does:**
- Prints seed provenance for the train/test split and model `random_state`.
- Runs integrity checks on the dataset (feature version, feature count, NaN/inf,
  label values, and `groups` shape).
- Prints a dataset summary: rows, positive rate, per-source counts, ALNS ratio,
  and number of grouped placement decisions.
- Uses a grouped holdout split and grouped cross-validation when `groups` are present.
- Trains with class-balanced sample weights and early stopping.
- Evaluates row-wise ROC-AUC / Average Precision and grouped ranking metrics
  (`top1_accuracy`, mean reciprocal rank).
- Checks quality gates and warns if scores are below the thresholds.
- Saves `repair_model_v1.pkl` with the model, scaler, metrics, quality gates,
  group metadata, and seed provenance.

**Quality thresholds**: ROC-AUC ≥ 0.80, Average Precision ≥ 0.60,
Top-1 repair accuracy ≥ 0.70.


In [ ]:
train_repair_model(
    TrainRepairModelConfig(
        data=[str(DATA_DIR / "synthetic_v1.pkl")],
        output=str(TRAINING_DIR / "repair_model_v1.pkl"),
        seed=SEED,
        min_roc_auc=0.80,
        min_average_precision=0.60,
        min_top1_accuracy=0.70,
        require_alns_states=False,   # v1 baseline — ALNS data not yet available
        cv_folds=5,
        no_learning_curves=True,
        no_plots=True,
    )
)


## Step 3 — Collect ALNS repair states (covariate-shift mitigation)

The baseline model is trained on BFD-labelled synthetic BFD-prefix states. During
actual ALNS search, the solver visits partially destroyed solutions with residual
bin-load patterns that may not appear in pure BFD construction. This **covariate
shift** can degrade repair quality.

`collect_alns_states` runs the v1 model on 500 fresh instances, samples realistic
ALNS destroy moves (random-item, worst-load, and related-item removal), captures
every repair decision it encounters, and re-labels those states with the BFD
oracle. The captured rows also carry `groups`, so each repair decision remains a
single grouped ranking example.

This is a single-pass **DAgger-lite** approach: cheaper than true DAgger, but it
reduces the distribution gap between offline training and online ALNS repair.

**Expected output**: row count, positive rate, and a saved payload containing
`X`, `y`, `groups`, `feature_version`, `source='alns_states'`, and `summary`.
The `source` tag lets the training script reliably count and report the ALNS
proportion.


In [ ]:
collect_alns_states(
    CollectAlnsStatesConfig(
        model_path=str(TRAINING_DIR / "repair_model_v1.pkl"),
        instances=500,
        n_min=50,
        n_max=200,
        max_negatives=5,
        iterations=200,
        seed=ALNS_SEED,
        output=str(DATA_DIR / "alns_states_v1.pkl"),
    )
)


## Step 4 — Generate supplementary synthetic data and retrain (v2)

Generates a smaller supplementary grouped synthetic dataset (`synthetic_v2.pkl`,
2 000 instances) and retrains by merging it with the grouped ALNS states collected
in Step 3.

The v2 training enforces `require_alns_states=True`: it raises immediately if no
ALNS-tagged dataset is present, making the covariate-shift requirement explicit
and impossible to accidentally skip.

**Expected output**:
- Dataset summary with the ALNS ratio (ALNS rows / total rows) printed.
- Group summary showing grouped placement decisions are available.
- Covariate-shift gate: ALNS rows found and accepted.
- Grouped split/CV output (`Train groups`, `Test groups`, CV scores).
- Quality gate results for ROC-AUC, Average Precision, and Top-1 repair accuracy.
- Model saved as `repair_model_v2.pkl` with the full provenance bundle.


In [ ]:
# 4a. Generate the supplementary synthetic dataset
generate_dataset(
    GenerateDatasetConfig(
        instances=2000,
        n_min=50,
        n_max=200,
        max_negatives=3,
        seed=SYNTHETIC_V2_SEED,
        workers=1,
        output=str(DATA_DIR / "synthetic_v2.pkl"),
    )
)

# 4b. Retrain with synthetic + ALNS-state data
train_repair_model(
    TrainRepairModelConfig(
        data=[
            str(DATA_DIR / "synthetic_v2.pkl"),
            str(DATA_DIR / "alns_states_v1.pkl"),
        ],
        output=str(TRAINING_DIR / "repair_model_v2.pkl"),
        seed=SEED,
        min_roc_auc=0.80,
        min_average_precision=0.60,
        min_top1_accuracy=0.70,
        require_alns_states=True,   # v2: raises if no ALNS data detected
        cv_folds=3,
        no_learning_curves=True,
        no_plots=True,
    )
)


## Step 5 — Optional benchmark

Run the Falkenauer-U benchmark to measure end-to-end solver quality with the
newly trained v2 model.  Set `RUN_BENCHMARK = True` to execute.

This step is intentionally disabled by default because it can take 10–30 minutes
depending on the runtime and instance size.


In [ ]:
RUN_BENCHMARK = False

if RUN_BENCHMARK:
    from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns import (
        hybrid_alns_solver,
    )
    from bin_packing_optimization.utilities.benchmarking import create_benchmark

    benchmark = create_benchmark(
        dataset_key="falkenauer-u",
        solver_module=hybrid_alns_solver,
        time_limit=None,
    )
    benchmark.run(method=None, method_args={"max_iterations": 500})
    csv_path = benchmark.save_results_to_csv()
    print("Results saved to:", csv_path)

## Validation checklist

After running all cells, verify the following in the printed output:

- [ ] **Reproducibility**: seed constants printed at setup match the values defined above
- [ ] **Integrity**: all dataset files pass checks for feature version, feature count, NaN/inf, labels, and `groups` shape
- [ ] **Grouped data**: generated and collected payloads include `groups`; training reports grouped placement-decision counts
- [ ] **Dataset summary**: rows, positive rate, per-source counts, and ALNS ratio printed for each training run
- [ ] **Grouped validation**: training reports `Train groups`, `Test groups`, and grouped CV scores
- [ ] **ALNS ratio**: v2 training reports `ALNS rows: N (X.X% of total)`
- [ ] **Covariate-shift gate**: `require_alns_states=True` accepted without error
- [ ] **Quality gates**: `✓ ROC-AUC ≥ 0.80`, `✓ Average Precision ≥ 0.60`, and `✓ Top-1 repair accuracy ≥ 0.70`
- [ ] **Model saved**: `repair_model_v1.pkl` and `repair_model_v2.pkl` present in `TRAINING_DIR`
